In [56]:
import os
import pandas
import json
import joblib
import base64
import sys
import yaml
import logging
from typing import Any
from pathlib import Path
from box import ConfigBox
from sqlalchemy import create_engine
from ensure import ensure_annotations
from box.exceptions import BoxValueError
from dataclasses import dataclass
import urllib.request as request
from zipfile import ZipFile
import pandas as pd
import time


In [57]:
%pwd
os.chdir('G:/Data Analytics Project/Vendor_Performance')

In [58]:
%pwd

'G:\\Data Analytics Project\\Vendor_Performance'

In [59]:
CONFIG_FILE_PATH = Path("config/config.yaml")

In [60]:
logging_str = "[%(asctime)s: %(levelname)s: %(module)s: %(message)s]"

log_dir = "logs"
log_filepath = os.path.join(log_dir,"Vendor_Performance.log")
os.makedirs(log_dir, exist_ok=True)


logging.basicConfig(
    level= logging.INFO,
    format= logging_str,

    handlers=[
        logging.FileHandler(log_filepath),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger("Vendor_Performance")

In [61]:

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    file_extension: str

@dataclass(frozen=True)
class InventoryConfig:
    root_dir: Path
    database_name: Path

In [62]:
@ensure_annotations
def read_yaml(path_to_yaml: Path) -> ConfigBox:
    """reads yaml file and returns
    Args:
        path_to_yaml (str): path like input
    Raises:
        ValueError: if yaml file is empty
        e: empty file
    Returns:
        ConfigBox: ConfigBox type
    """
    try:
        with open(path_to_yaml, 'r') as yaml_file:
            content = yaml.safe_load(yaml_file)
            logger.info(f"yaml file: {path_to_yaml} loaded successfully")
            print(ConfigBox(content))
            return ConfigBox(content)
    except BoxValueError:
        raise ValueError("yaml is Empty")
    except Exception as e:
        raise e
    
@ensure_annotations
def create_directories(path_to_directories: list, verbose=True):
    """create list of directories

    Args:
        path_to_directories (list): list of path of directories
        ignore_log (bool, optional): ignore if multiple dirs is to be created. Defaults to False.
    """
    try:
        for path in path_to_directories:
            os.makedirs(path, exist_ok=True)
            if verbose:
                logger.info(f"created directory at: {path}")
    except Exception as e:
        raise e
    
@ensure_annotations
def get_size(path: Path) -> str:
    """get size in KB

    Args:
        path (Path): path of the file

    Returns:
        str: size in KB
    """
    size_in_kb = round(os.path.getsize(path)/1024)
    return f"~ {size_in_kb} KB"



In [63]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH):
    
        self.config = read_yaml(config_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            file_extension=config.file_extension 
        )

        return data_ingestion_config
    
    def get_inventory_data_config(self) -> InventoryConfig:
        config = self.config.inventory_data_db

        create_directories([config.root_dir])
        
        inventory_data_config = InventoryConfig(
            root_dir=config.root_dir,
            database_name=config.database_name
        )

        return inventory_data_config


In [64]:
class Inventory_DB:
    def __init__(self, config: InventoryConfig):
        self.config = config

    def create_inventory_db_file(self):
        file_path = self.config.database_name
        filedir, _ = os.path.split(file_path)

        if not os.path.exists(filedir):  
            create_directories([filedir])
            logging.info(f"Created directory at: {file_path}")
        else:
            logging.info(f"directory already exists: {file_path}")
        
        if not os.path.exists(file_path):
            with open(file_path, 'w') as f:
                pass
            logging.info(f"Created empty file: {file_path}")
        else:
            logging.info(f"File already exists: {file_path}")

In [65]:
config = ConfigurationManager()
inventory_data_config = config.get_inventory_data_config()
inventory = Inventory_DB(config=inventory_data_config)
inventory.create_inventory_db_file()

[2026-03-26 04:13:33,469: INFO: 4105043225: yaml file: config\config.yaml loaded successfully]
{'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data', 'file_extension': '.csv'}, 'inventory_data_db': {'root_dir': 'artifacts/inventoryDB', 'database_name': 'artifacts/inventoryDB/inventory_data.db'}}
[2026-03-26 04:13:33,476: INFO: 4105043225: created directory at: artifacts]
[2026-03-26 04:13:33,477: INFO: 4105043225: created directory at: artifacts/inventoryDB]
[2026-03-26 04:13:33,478: INFO: 1587978096: directory already exists: artifacts/inventoryDB/inventory_data.db]
[2026-03-26 04:13:33,478: INFO: 1587978096: File already exists: artifacts/inventoryDB/inventory_data.db]


In [ ]:
def ingest_db(df, table_name, db_path, engine):
    if os.path.exists(db_path): 
        # df.to_sql(table_name, con=engine, if_exists= 'replace', index = False)
        return True

In [67]:
def load_raw_data():
    # Obtaining Data from the config.yaml file
    config = read_yaml(CONFIG_FILE_PATH)
    f = config.data_ingestion.root_dir
    
    # obtaining the extension from the config.yaml file
    ext = config.data_ingestion.file_extension

    # obtaining the database file from the config.yaml file
    db_path = config.inventory_data_db.database_name

    start = time.time()
    for file in os.listdir(f):

        if file.endswith(ext):
            # Joining files path
            file_path = os.path.join(f, file)
            
            # 2. Read the file
            df = pd.read_csv(file_path)

            logger.info(f"Displaying File Shape {file}: {df.shape}")
            
            # engine is initialized
            engine = create_engine(f'sqlite:///{db_path}')

            logger.info(f"Ingesting {file} into Database: {db_path}")

            # Ingest into SQL
            ingest_db(df, file[:-4], db_path, engine)

    end = time.time()
    total_time = (end - start) / 60
    logger.info(f"--------------------------Ingestion Complete--------------------------")
    logger.info(f"Total Time Taken {total_time} in minutes")


if __name__ =='__main__':
    load_raw_data()

[2026-03-26 04:13:33,513: INFO: 4105043225: yaml file: config\config.yaml loaded successfully]
{'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data', 'file_extension': '.csv'}, 'inventory_data_db': {'root_dir': 'artifacts/inventoryDB', 'database_name': 'artifacts/inventoryDB/inventory_data.db'}}
[2026-03-26 04:13:33,843: INFO: 2248167240: Displaying File Shape begin_inventory.csv: (206529, 9)]
[2026-03-26 04:13:33,849: INFO: 2248167240: Ingesting begin_inventory.csv into Database: artifacts/inventoryDB/inventory_data.db]
[2026-03-26 04:13:34,252: INFO: 2248167240: Displaying File Shape end_inventory.csv: (224489, 9)]
[2026-03-26 04:13:34,253: INFO: 2248167240: Ingesting end_inventory.csv into Database: artifacts/inventoryDB/inventory_data.db]
[2026-03-26 04:13:39,768: INFO: 2248167240: Displaying File Shape purchases.csv: (2372474, 16)]
[2026-03-26 04:13:39,770: INFO: 2248167240: Ingesting purchases.csv into Database: artifacts/inventoryDB/inventory_data.db]
[

## 📊 Exploratory Data Analysis

Understanding the dataset to examine how data is structured and stored in the database. This step helps determine whether creating aggregated tables is necessary to support business insights and decision-making.

### 🎯 Objectives

- Analyze how data is organized within the database  
- Identify the need for aggregated tables to improve analysis and performance  



In [68]:
# Importing important libraries for EDA

import sqlite3
from pathlib import Path

In [69]:
config = read_yaml(CONFIG_FILE_PATH)

# obtaining the database file from the config.yaml file
db_path = config.inventory_data_db.database_name

conn = sqlite3.connect(f'{db_path}')

[2026-03-26 04:13:58,846: INFO: 4105043225: yaml file: config\config.yaml loaded successfully]
{'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data', 'file_extension': '.csv'}, 'inventory_data_db': {'root_dir': 'artifacts/inventoryDB', 'database_name': 'artifacts/inventoryDB/inventory_data.db'}}


In [70]:
import pandas as pd

def Query(table_name: str, column_name: list = "*", vendor_number: int = None, **args):

    if isinstance(column_name, list):
        column_str = ", ".join(column_name)
    else:
        # This handles the default "*" or a single string
        column_str = column_name
    
    # query logic
    if not vendor_number:
        query = f"SELECT {column_str} FROM {table_name}"
        params = None
    else:
        query = f"SELECT {column_str} FROM {table_name} WHERE VendorNumber= {vendor_number}"
        # params = (vendor_number,)
        
    return pd.read_sql_query(query, conn)

In [71]:
tables = Query('sqlite_master')
tables

,type,name,tbl_name,rootpage,sql
0,table,vendor_sales_summary,vendor_sales_summary,103615,"CREATE TABLE vendor_sales_summary (\n\t""Vendor..."
1,table,begin_inventory,begin_inventory,2,"CREATE TABLE begin_inventory (\n\t""InventoryId..."
2,table,end_inventory,end_inventory,4656,"CREATE TABLE end_inventory (\n\t""InventoryId"" ..."
3,table,purchases,purchases,9717,"CREATE TABLE purchases (\n\t""InventoryId"" TEXT..."
4,table,purchase_prices,purchase_prices,103198,"CREATE TABLE purchase_prices (\n\t""Brand"" BIGI..."
5,table,sales,sales,103482,"CREATE TABLE sales (\n\t""InventoryId"" TEXT, \n..."
6,table,vendor_invoice,vendor_invoice,103483,"CREATE TABLE vendor_invoice (\n\t""VendorNumber..."


In [72]:
for t in tables['name']:
    query = f"Select count(*) as count from {t}"
    print('-'*50, f'{t}', '-'*50)
    print(f"Count of Records: {pd.read_sql(query, conn)['count'][0]}")
    display(
        pd.read_sql(f"SELECT * from {t} limit 10", conn)
    )



-------------------------------------------------- vendor_sales_summary --------------------------------------------------
Count of Records: 10692


,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesPrice,TotalSalesDollars,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalestoPurchaseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750.0,145080,3811251.60,142049.0,672819.31,5101919.51,260999.20,68601.68,1290667.91,25.297693,0.979108,1.338647
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750.0,164038,3804041.22,160247.0,561512.37,4819073.49,294438.66,144929.24,1015032.27,21.062810,0.976890,1.266830
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750.0,187407,3418303.68,187140.0,461140.15,4538120.60,343854.07,123780.22,1119816.92,24.675786,0.998575,1.327594
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750.0,201682,3261197.94,200412.0,420050.01,4475972.88,368242.80,257032.07,1214774.94,27.139908,0.993703,1.372493
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750.0,138109,3023206.01,135838.0,545778.28,4223107.62,249587.83,257032.07,1199901.61,28.412764,0.983556,1.396897
5,480,BACARDI USA INC,3858,Grey Goose Vodka,17.77,23.99,750.0,138809,2466635.93,141860.0,446932.09,3383912.40,111699.19,89286.27,917276.47,27.106980,1.021980,1.371873
6,17035,PERNOD RICARD USA,2589,Jameson Irish Whiskey,30.76,39.99,1750.0,70783,2177285.08,69627.0,614529.34,2773367.73,127931.67,123780.22,596082.65,21.493098,0.983668,1.273773
7,3960,DIAGEO NORTH AMERICA INC,3102,Smirnoff Traveler,12.94,17.99,1750.0,161386,2088334.84,148265.0,292586.29,2592041.35,272422.60,257032.07,503706.51,19.432812,0.918698,1.241200
8,3960,DIAGEO NORTH AMERICA INC,3489,Tanqueray,20.73,27.99,1750.0,91835,1903739.55,90481.0,503661.02,2640491.19,166244.44,257032.07,736751.64,27.902068,0.985256,1.387002
9,12546,JIM BEAM BRANDS COMPANY,1376,Jim Beam,16.29,21.99,1750.0,108866,1773427.14,107061.0,426902.78,2435393.39,196707.35,123880.97,661966.25,27.181081,0.983420,1.373269


-------------------------------------------------- begin_inventory --------------------------------------------------
Count of Records: 206529


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,startDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,8,12.99,2024-01-01
1,1_HARDERSFIELD_60,1,HARDERSFIELD,60,Canadian Club 1858 VAP,750mL,7,10.99,2024-01-01
2,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,6,36.99,2024-01-01
3,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,3,38.99,2024-01-01
4,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,6,34.99,2024-01-01
5,1_HARDERSFIELD_75,1,HARDERSFIELD,75,Three Olives Tomato Vodka,750mL,18,14.99,2024-01-01
6,1_HARDERSFIELD_77,1,HARDERSFIELD,77,Three Olives Espresso Vodka,750mL,7,14.99,2024-01-01
7,1_HARDERSFIELD_79,1,HARDERSFIELD,79,Three Olives Loopy Vodka,750mL,2,14.99,2024-01-01
8,1_HARDERSFIELD_115,1,HARDERSFIELD,115,Belvedere Vodka,Liter,5,27.99,2024-01-01
9,1_HARDERSFIELD_120,1,HARDERSFIELD,120,Tarantula Azul Tequila Gift,750mL,11,13.99,2024-01-01


-------------------------------------------------- end_inventory --------------------------------------------------
Count of Records: 224489


...

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,58,Gekkeikan Black & Gold Sake,12.99,750mL,750,1,9.28,8320,SHAW ROSS INT L IMP LTD
1,62,Herradura Silver Tequila,36.99,750mL,750,1,28.67,1128,BROWN-FORMAN CORP
2,63,Herradura Reposado Tequila,38.99,750mL,750,1,30.46,1128,BROWN-FORMAN CORP
3,72,No. 3 London Dry Gin,34.99,750mL,750,1,26.11,9165,ULTRA BEVERAGE COMPANY LLP
4,75,Three Olives Tomato Vodka,14.99,750mL,750,1,10.94,7245,PROXIMO SPIRITS INC.
5,77,Three Olives Espresso Vodka,12.99,750mL,750,1,10.39,7245,PROXIMO SPIRITS INC.
6,79,Three Olives Loopy Vodka,14.99,750mL,750,1,9.62,7245,PROXIMO SPIRITS INC.
7,115,Belvedere Vodka,27.99,1000mL,1000,1,21.37,8112,MOET HENNESSY USA INC
8,126,Grey Goose Vodka,32.99,1000mL,1000,1,20.14,480,BACARDI USA INC
9,168,Three Olives Strawberry,12.99,750mL,750,1,8.95,7245,PROXIMO SPIRITS INC.


-------------------------------------------------- sales --------------------------------------------------
Count of Records: 12825363


,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-01,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,2024-01-02,750.0,1,1.57,12546,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-03,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
3,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,14.49,14.49,2024-01-08,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
4,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,2,69.98,34.99,2024-01-09,375.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
5,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,1,34.99,34.99,2024-01-15,375.0,1,0.39,12546,JIM BEAM BRANDS COMPANY
6,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,1,34.99,34.99,2024-01-22,375.0,1,0.39,12546,JIM BEAM BRANDS COMPANY
7,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,1,34.99,34.99,2024-01-30,375.0,1,0.39,12546,JIM BEAM BRANDS COMPANY
8,1_HARDERSFIELD_10058,1,10058,F Coppola Dmd Ivry Cab Svgn,750mL,4,59.96,14.99,2024-01-05,750.0,2,0.45,2000,SOUTHERN WINE & SPIRITS NE
9,1_HARDERSFIELD_10058,1,10058,F Coppola Dmd Ivry Cab Svgn,750mL,1,14.99,14.99,2024-01-06,750.0,2,0.11,2000,SOUTHERN WINE & SPIRITS NE


-------------------------------------------------- vendor_invoice --------------------------------------------------
Count of Records: 5543


,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,None
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,None
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,None
5,2396,BLACK PRINCE DISTILLERY INC,2024-01-08,8191,2023-12-25,2024-02-06,23,234.83,2.30,None
6,1128,BROWN-FORMAN CORP,2024-01-09,8150,2023-12-23,2024-02-19,4684,65403.57,1808.77,None
7,1189,BULLY BOY DISTILLERS,2024-01-09,8171,2023-12-24,2024-02-04,6,132.30,5.29,None
8,1273,CALEDONIA SPIRITS INC,2024-01-06,8172,2023-12-24,2024-02-15,5,146.80,15.53,None
9,11567,CAMPARI AMERICA,2024-01-06,8151,2023-12-23,2024-02-20,1321,12039.71,398.71,None


# Comparing Purchases and Purchases_Prices

In [73]:
purchases = Query('purchases')
purchases

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2372469,49_GARIGILL_22298,49,22298,Zorvino Vyds Sangiovese,750mL,90058,ZORVINO VINEYARDS,13593,2024-12-19,2024-12-28,2025-01-09,2025-02-06,8.06,12,96.72,2
2372470,1_HARDERSFIELD_19556,1,19556,Zorvino Bacca Z Blackberry,750mL,90058,ZORVINO VINEYARDS,13593,2024-12-19,2024-12-27,2025-01-09,2025-02-06,9.39,12,112.68,2
2372471,66_EANVERNESS_22297,66,22297,Zorvino Vyds Pearz,750mL,90058,ZORVINO VINEYARDS,13593,2024-12-19,2024-12-26,2025-01-09,2025-02-06,6.75,12,81.00,2
2372472,69_MOUNTMEND_19557,69,19557,Zorvino Fragole Z Strawberry,750mL,90058,ZORVINO VINEYARDS,13593,2024-12-19,2024-12-26,2025-01-09,2025-02-06,9.39,12,112.68,2


In [74]:
purchases.groupby(['Brand', 'PurchasePrice'])[['Quantity', 'Dollars']].sum()

,,Quantity,Dollars
Brand,PurchasePrice,,
58,9.28,3550,32944.00
60,7.40,1633,12084.20
61,10.60,312,3307.20
62,28.67,3200,91744.00
63,30.46,2855,86963.30
...,...,...,...
90089,77.92,32,2493.44
90090,448.27,6,2689.62
90604,78.42,118,9253.56


In [75]:
purchase_prices = Query('purchase_prices')
purchase_prices

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,58,Gekkeikan Black & Gold Sake,12.99,750mL,750,1,9.28,8320,SHAW ROSS INT L IMP LTD
1,62,Herradura Silver Tequila,36.99,750mL,750,1,28.67,1128,BROWN-FORMAN CORP
2,63,Herradura Reposado Tequila,38.99,750mL,750,1,30.46,1128,BROWN-FORMAN CORP
3,72,No. 3 London Dry Gin,34.99,750mL,750,1,26.11,9165,ULTRA BEVERAGE COMPANY LLP
4,75,Three Olives Tomato Vodka,14.99,750mL,750,1,10.94,7245,PROXIMO SPIRITS INC.
...,...,...,...,...,...,...,...,...,...
12256,44917,Ferreira 10-Yr Tawny Port,24.99,750mL,750,2,16.55,90024,VINILANDIA USA
12257,44944,Sanford Santa Rita Pnt Nr,22.99,750mL,750,2,14.93,4425,MARTIGNETTI COMPANIES
12258,45016,Neal One Lane Bridg Cab Svgn,93.99,750mL,750,2,61.43,10754,PERFECTA WINES
12259,46011,Folonari Pnt Nr Venezie,12.99,1500ml,1500,2,8.90,9744,FREDERICK WILDMAN & SONS


In [76]:
profit_margin = (purchase_prices['Price'] - purchase_prices['PurchasePrice'])/purchase_prices['Price']
purchase_prices['profit_margin'] = profit_margin 
purchase_prices.columns

Index(['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification',
       'PurchasePrice', 'VendorNumber', 'VendorName', 'profit_margin'],
      dtype='str')

In [77]:
purchase_prices.groupby(['Brand', 'Price'])[['PurchasePrice', 'profit_margin']].sum()

,,PurchasePrice,profit_margin
Brand,Price,,
58,12.99,9.28,0.285604
60,10.99,7.40,0.326661
61,13.99,10.60,0.242316
62,36.99,28.67,0.224926
63,38.99,30.46,0.218774
...,...,...,...
90090,649.99,448.27,0.310343
90590,19.95,13.12,0.342356
90604,119.99,78.42,0.346446


# Vendor Invoice

In [78]:
vendor_invoice = Query('vendor_invoice',vendor_number=4466)
vendor_invoice

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.33,16.97,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20,3.30,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,2024-02-09,8604,2024-01-24,2024-03-15,347,3261.37,17.61,None
6,4466,AMERICAN VINTAGE BEVERAGE,2024-02-17,8793,2024-02-05,2024-04-02,72,675.36,3.17,None
7,4466,AMERICAN VINTAGE BEVERAGE,2024-03-01,8892,2024-02-12,2024-03-28,117,1096.05,5.15,None
8,4466,AMERICAN VINTAGE BEVERAGE,2024-03-07,8995,2024-02-19,2024-04-02,129,1209.27,5.44,None
9,4466,AMERICAN VINTAGE BEVERAGE,2024-03-12,9033,2024-02-22,2024-04-16,147,1377.87,6.61,None


In [79]:
#  Checking the Uniquness of PONumber i.e. Purchased-Order-Number
vendor_invoice['PONumber'].nunique()
vendor_invoice.shape

(55, 10)

# Sales

In [80]:
sales = Query('sales')
sales.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity',
       'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification',
       'ExciseTax', 'VendorNo', 'VendorName'],
      dtype='str')

In [81]:
sales.head(10)

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-01,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,2024-01-02,750.0,1,1.57,12546,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-03,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
3,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,14.49,14.49,2024-01-08,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
4,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,2,69.98,34.99,2024-01-09,375.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
5,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,1,34.99,34.99,2024-01-15,375.0,1,0.39,12546,JIM BEAM BRANDS COMPANY
6,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,1,34.99,34.99,2024-01-22,375.0,1,0.39,12546,JIM BEAM BRANDS COMPANY
7,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,1,34.99,34.99,2024-01-30,375.0,1,0.39,12546,JIM BEAM BRANDS COMPANY
8,1_HARDERSFIELD_10058,1,10058,F Coppola Dmd Ivry Cab Svgn,750mL,4,59.96,14.99,2024-01-05,750.0,2,0.45,2000,SOUTHERN WINE & SPIRITS NE
9,1_HARDERSFIELD_10058,1,10058,F Coppola Dmd Ivry Cab Svgn,750mL,1,14.99,14.99,2024-01-06,750.0,2,0.11,2000,SOUTHERN WINE & SPIRITS NE


In [82]:
sales.groupby(['Brand'])[['SalesDollars', 'SalesPrice', 'SalesQuantity']].sum()

,SalesDollars,SalesPrice,SalesQuantity
Brand,,,
58,43341.54,28145.64,3446
60,18716.25,10720.79,1775
61,4364.88,363.74,312
62,119863.75,90154.51,3125
63,112249.22,88553.10,2778
...,...,...,...
90089,19078.41,5759.52,159
90090,9749.85,4549.93,15
90604,9119.24,2639.78,76


# Analysis

· The purchases table contains actual purchase data, including the date of purchase, products (brands) purchased by vendors, the amount paid (in dollars),
and the quantity purchased.

· The purchase price column is derived from the purchase_prices table, which provides product-wise actual and purchase prices. The combination of vendor
and brand is unique in this table.

· The vendor_invoice table aggregates data from the purchases table, summarizing quantity and dollar amounts, along with an additional column for freight.
This table maintains uniqueness based on vendor and PO number.

· The sales table captures actual sales transactions, detailing the brands purchased by vendors, the quantity sold, the selling price, and the revenue earned.

As the data that we need for analysis is distributed in different tables, we need to create a summary table containing:

. purchase transactions made by vendors
· sales transaction data
· freight costs for each vendor
· actual product prices from vendors

In [83]:
query = "Select VendorNumber, SUM(Freight) as FreightCost from vendor_invoice GROUP BY VendorNumber"

vendor_invoice_summary = pd.read_sql_query(query, conn)
vendor_invoice_summary

,VendorNumber,FreightCost
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19
...,...,...
121,98450,856.02
122,99166,130.09
123,172662,178.34
124,173357,202.50


In [84]:
print("Purchase_prices Column")
print(purchase_prices.columns)
print("\n")
print("Purchases Column")
print(purchases.columns)


Purchase_prices Column
Index(['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification',
       'PurchasePrice', 'VendorNumber', 'VendorName', 'profit_margin'],
      dtype='str')


Purchases Column
Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber',
       'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate',
       'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification'],
      dtype='str')


In [85]:
# p -> pruchases  and pp -> purchase_prices

query = """
SELECT 
p.VendorName, 
p.VendorNumber, 
p.Brand, 
p.Description,
p.PurchasePrice, 
pp.Volume,
pp.Price as ActualPrice, 
SUM(p.Quantity) as TotalPurchasedQuantity,  
SUM(p.Dollars) as TotalPurchaseDollar 
from purchases p JOIN purchase_prices pp
ON p.Brand = pp.Brand
WHERE p.PurchasePrice > 0
Group By  p.VendorName, p.VendorNumber, p.Brand
ORDER BY TotalPurchaseDollar
"""
purchase_summary = pd.read_sql_query(query, conn)
purchase_summary

,VendorName,VendorNumber,Brand,Description,PurchasePrice,Volume,ActualPrice,TotalPurchasedQuantity,TotalPurchaseDollar
0,PROXIMO SPIRITS INC.,7245,3065,Three Olives Grape Vodka,0.71,50,0.99,1,0.71
1,DIAGEO NORTH AMERICA INC,3960,6127,The Club Strawbry Margarita,1.47,200,1.99,1,1.47
2,HEAVEN HILL DISTILLERIES,3924,9123,Deep Eddy Vodka,0.74,50,0.99,2,1.48
3,SAZERAC CO INC,8004,5683,Dr McGillicuddy's Apple Pie,0.39,50,0.49,6,2.34
4,WINE GROUP INC,9815,8527,Concannon Glen Ellen Wh Zin,1.32,750,4.99,2,2.64
...,...,...,...,...,...,...,...,...,...
10687,DIAGEO NORTH AMERICA INC,3960,3545,Ketel One Vodka,21.89,1750,29.99,138109,3023206.01
10688,DIAGEO NORTH AMERICA INC,3960,4261,Capt Morgan Spiced Rum,16.17,1750,22.99,201682,3261197.94
10689,PERNOD RICARD USA,17035,8068,Absolut 80 Proof,18.24,1750,24.99,187407,3418303.68
10690,MARTIGNETTI COMPANIES,4425,3405,Tito's Handmade Vodka,23.19,1750,28.99,164038,3804041.22


In [86]:
print("Sales Column")
print(sales.columns)

Sales Column
Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity',
       'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification',
       'ExciseTax', 'VendorNo', 'VendorName'],
      dtype='str')


In [87]:
query = """
SELECT 
VendorNo, 
Brand, 
Description,
SUM(SalesDollars) as TotalSalesDollar,
SUM(SalesPrice) as TotalSalesPrice,
SUM(SalesQuantity) as TotalSalesQuantity,
SUM(ExciseTax) as TotalExciseTax
from sales
Group By  VendorNo, Brand
ORDER BY TotalSalesDollar
"""
sales_summary = pd.read_sql_query(query, conn)
sales_summary

,VendorNo,Brand,Description,TotalSalesDollar,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax
0,8004,5287,Dr McGillicuddy's Vanilla,0.98,0.98,2,0.10
1,9206,2773,Revel Stoke Roasted Pecan,0.99,0.99,1,0.05
2,3252,3933,New Amsterdam Vodka,1.98,0.99,2,0.10
3,3924,9123,Deep Eddy Vodka,1.98,0.99,2,0.10
4,10050,3623,Spud Potato Vodka,1.98,1.98,2,0.10
...,...,...,...,...,...,...,...
11267,3960,3545,Ketel One Vodka,4223107.62,545778.28,135838,249587.83
11268,3960,4261,Capt Morgan Spiced Rum,4475972.88,420050.01,200412,368242.80
11269,17035,8068,Absolut 80 Proof,4538120.60,461140.15,187140,343854.07
11270,4425,3405,Tito's Handmade Vodka,4819073.49,561512.37,160247,294438.66


## Combining Both Sales_Summary and Purchase_Summary

This query aggregates sales, purchases, and freight data separately, then combines them to create a vendor–brand level summary report.
It ensures accurate calculations and better performance by avoiding duplicate data during joins.


In [88]:
query = """
WITH sales_summary AS (
    SELECT 
        VendorNo, 
        Brand, 
        SUM(SalesDollars) AS TotalSalesDollar,
        SUM(SalesPrice) AS TotalSalesPrice,
        SUM(SalesQuantity) AS TotalSalesQuantity,
        SUM(ExciseTax) AS TotalExciseTax
    FROM sales
    GROUP BY VendorNo, Brand
),

purchase_summary AS (
    SELECT 
        p.VendorName, 
        p.VendorNumber, 
        p.Brand,
        p.Description, 
        MAX(p.PurchasePrice) AS PurchasePrice,  
        MAX(pp.Price) AS ActualPrice,
        MAX(pp.Volume) AS Volume,
        SUM(p.Quantity) AS TotalPurchasedQuantity,  
        SUM(p.Dollars) AS TotalPurchaseDollar 
    FROM purchases p 
    JOIN purchase_prices pp
        ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorName, p.VendorNumber, p.Brand
),

vendor_invoice_summary AS (
    SELECT 
        VendorNumber, 
        SUM(Freight) AS FreightCost 
    FROM vendor_invoice 
    GROUP BY VendorNumber
)

SELECT 
    ps.VendorName,
    ps.VendorNumber,
    ps.Brand,
    ps.Description,
    ps.PurchasePrice,
    ps.ActualPrice,
    ps.Volume,

    ps.TotalPurchasedQuantity,
    ps.TotalPurchaseDollar,

    ss.TotalSalesDollar,
    ss.TotalSalesPrice,
    ss.TotalSalesQuantity,
    ss.TotalExciseTax,

    vis.FreightCost

FROM purchase_summary ps

LEFT JOIN sales_summary ss
    ON ps.VendorNumber = ss.VendorNo
    AND ps.Brand = ss.Brand

LEFT JOIN vendor_invoice_summary vis
    ON ps.VendorNumber = vis.VendorNumber

ORDER BY ps.TotalPurchaseDollar DESC;

"""
final_summary = pd.read_sql_query(query, conn)

In [89]:
final_summary

,VendorName,VendorNumber,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchasedQuantity,TotalPurchaseDollar,TotalSalesDollar,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax,FreightCost
0,BROWN-FORMAN CORP,1128,1233,Jack Daniels No 7 Black,26.27,36.99,1750,145080,3811251.60,5101919.51,672819.31,142049.0,260999.20,68601.68
1,MARTIGNETTI COMPANIES,4425,3405,Tito's Handmade Vodka,23.19,28.99,1750,164038,3804041.22,4819073.49,561512.37,160247.0,294438.66,144929.24
2,PERNOD RICARD USA,17035,8068,Absolut 80 Proof,18.24,24.99,1750,187407,3418303.68,4538120.60,461140.15,187140.0,343854.07,123780.22
3,DIAGEO NORTH AMERICA INC,3960,4261,Capt Morgan Spiced Rum,16.17,22.99,1750,201682,3261197.94,4475972.88,420050.01,200412.0,368242.80,257032.07
4,DIAGEO NORTH AMERICA INC,3960,3545,Ketel One Vodka,21.89,29.99,1750,138109,3023206.01,4223107.62,545778.28,135838.0,249587.83,257032.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,WINE GROUP INC,9815,8527,Concannon Glen Ellen Wh Zin,1.32,4.99,750,2,2.64,15.95,10.96,5.0,0.55,27100.41
10688,SAZERAC CO INC,8004,5683,Dr McGillicuddy's Apple Pie,0.39,0.49,50,6,2.34,65.66,1.47,134.0,7.04,50293.62
10689,HEAVEN HILL DISTILLERIES,3924,9123,Deep Eddy Vodka,0.74,0.99,50,2,1.48,1.98,0.99,2.0,0.10,14069.87
10690,DIAGEO NORTH AMERICA INC,3960,6127,The Club Strawbry Margarita,1.47,1.99,200,1,1.47,143.28,77.61,72.0,15.12,257032.07


This query generates a vendor-wise sales and purchase summary, which is valuable for:

**Performance Optimization**: 

· The query involves heavy joins and aggregations on large datasets like sales and purchases. \
· Storing the pre-aggregated results avoids repeated expensive computations. \
· Helps in analyzing sales, purchases, and pricing for different vendors and brands. \
· Future Benefits of Storing this data for faster Dashboarding & Reporting. \
. Instead of running expensive queries each time, dashboards can fetch data quickly from vendor_sales_summary. 

### 🔍 Data Quality Check

Before performing further analysis, we validate the final table to ensure the data is clean and free from inconsistencies. \
This step helps identify issues like missing values, duplicates, or mismatched totals, ensuring reliable insights in later analysis.

In [90]:
final_summary.shape

(10692, 14)

1. Checking the data types in the final_summary DataFrame

In [91]:
final_summary.dtypes

VendorName                    str
VendorNumber                int64
Brand                       int64
Description                   str
PurchasePrice             float64
ActualPrice               float64
Volume                        str
TotalPurchasedQuantity      int64
TotalPurchaseDollar       float64
TotalSalesDollar          float64
TotalSalesPrice           float64
TotalSalesQuantity        float64
TotalExciseTax            float64
FreightCost               float64
dtype: object

`final_summary.dtypes` shows the data type of each column to help verify data consistency before analysis.


2. Check for missing values in each column to identify data gaps and ensure data quality before analysis

In [92]:
final_summary.isnull().sum()

VendorName                  0
VendorNumber                0
Brand                       0
Description                 0
PurchasePrice               0
ActualPrice                 0
Volume                      0
TotalPurchasedQuantity      0
TotalPurchaseDollar         0
TotalSalesDollar          178
TotalSalesPrice           178
TotalSalesQuantity        178
TotalExciseTax            178
FreightCost                 0
dtype: int64

### 📝 Missing Sales Values Explanation

`TotalSalesDollar`, `TotalSalesPrice`, `TotalSalesQuantity`, and `TotalExciseTax` are missing because these items were purchased but not yet sold.

In [102]:
final_summary['Volume'] = final_summary['Volume'].astype('float64')

In [104]:
final_summary.fillna(0, inplace=True)

,VendorName,VendorNumber,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchasedQuantity,TotalPurchaseDollar,TotalSalesDollar,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax,FreightCost
0,BROWN-FORMAN CORP,1128,1233,Jack Daniels No 7 Black,26.27,36.99,1750.0,145080,3811251.60,5101919.51,672819.31,142049.0,260999.20,68601.68
1,MARTIGNETTI COMPANIES,4425,3405,Tito's Handmade Vodka,23.19,28.99,1750.0,164038,3804041.22,4819073.49,561512.37,160247.0,294438.66,144929.24
2,PERNOD RICARD USA,17035,8068,Absolut 80 Proof,18.24,24.99,1750.0,187407,3418303.68,4538120.60,461140.15,187140.0,343854.07,123780.22
3,DIAGEO NORTH AMERICA INC,3960,4261,Capt Morgan Spiced Rum,16.17,22.99,1750.0,201682,3261197.94,4475972.88,420050.01,200412.0,368242.80,257032.07
4,DIAGEO NORTH AMERICA INC,3960,3545,Ketel One Vodka,21.89,29.99,1750.0,138109,3023206.01,4223107.62,545778.28,135838.0,249587.83,257032.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,WINE GROUP INC,9815,8527,Concannon Glen Ellen Wh Zin,1.32,4.99,750.0,2,2.64,15.95,10.96,5.0,0.55,27100.41
10688,SAZERAC CO INC,8004,5683,Dr McGillicuddy's Apple Pie,0.39,0.49,50.0,6,2.34,65.66,1.47,134.0,7.04,50293.62
10689,HEAVEN HILL DISTILLERIES,3924,9123,Deep Eddy Vodka,0.74,0.99,50.0,2,1.48,1.98,0.99,2.0,0.10,14069.87
10690,DIAGEO NORTH AMERICA INC,3960,6127,The Club Strawbry Margarita,1.47,1.99,200.0,1,1.47,143.28,77.61,72.0,15.12,257032.07


In [108]:
final_summary['VendorName'].str.strip()

0               BROWN-FORMAN CORP
1           MARTIGNETTI COMPANIES
2               PERNOD RICARD USA
3        DIAGEO NORTH AMERICA INC
4        DIAGEO NORTH AMERICA INC
                   ...           
10687              WINE GROUP INC
10688              SAZERAC CO INC
10689    HEAVEN HILL DISTILLERIES
10690    DIAGEO NORTH AMERICA INC
10691        PROXIMO SPIRITS INC.
Name: VendorName, Length: 10692, dtype: str

In [109]:
final_summary['VendorName'].unique()

<StringArray>
[          'BROWN-FORMAN CORP',       'MARTIGNETTI COMPANIES',
           'PERNOD RICARD USA',    'DIAGEO NORTH AMERICA INC',
             'BACARDI USA INC',     'JIM BEAM BRANDS COMPANY',
         'MAJESTIC FINE WINES',  'ULTRA BEVERAGE COMPANY LLP',
       'STOLI GROUP,(USA) LLC',        'PROXIMO SPIRITS INC.',
 ...
                    'UNCORKED',         'BRONCO WINE COMPANY',
     'MILTONS DISTRIBUTING CO',                'TRUETT HURST',
         'LAUREATE IMPORTS CO',     'FANTASY FINE WINES CORP',
 'AAPER ALCOHOL & CHEMICAL CO',      'SILVER MOUNTAIN CIDERS',
      'CAPSTONE INTERNATIONAL',          'FLAVOR ESSENCE INC']
Length: 128, dtype: str

In [110]:
final_summary['GrossProfit'] = final_summary['TotalSalesDollar'] - final_summary['TotalPurchaseDollar']

In [113]:
final_summary['ProfitMargin'] = (final_summary['GrossProfit'] / final_summary['TotalSalesDollar']) * 100

In [117]:
final_summary['StockTurnOver'] = final_summary['TotalSalesQuantity'] / final_summary['TotalPurchasedQuantity']

In [120]:
final_summary['SalestoPruchaseRatio'] = final_summary['TotalSalesDollar'] / final_summary['TotalPurchaseDollar']

In [123]:
final_summary.columns

Index(['VendorName', 'VendorNumber', 'Brand', 'Description', 'PurchasePrice',
       'ActualPrice', 'Volume', 'TotalPurchasedQuantity',
       'TotalPurchaseDollar', 'TotalSalesDollar', 'TotalSalesPrice',
       'TotalSalesQuantity', 'TotalExciseTax', 'FreightCost', 'GrossProfit',
       'ProfitMargin', 'StockTurnOver', 'SalestoPruchaseRatio'],
      dtype='str')

### 💾 Saving Final Data to Database

Now we store the cleaned and aggregated data into a new table for future use.

This allows faster querying, efficient reporting, and avoids recomputing complex aggregations repeatedly.

In [129]:
cursor = conn.cursor()

In [130]:
query = """
Create Table new_vendor_sales_summary (
VendorNumber INT,
VendorName VARCHAR(100),
Brand INT,
Description VARCHAR(100),
PurchasePrice DECIMAL(10,2),
ActualPrice DECIMAL(10,2),
Volume,
TotalPurchasedQuantity INT,
TotalPurchaseDollar DECIMAL(15, 2),
TotalSalesQuantity INT,
TotalSalesDollar DECIMAL(15, 2),
TotalSalesPrice DECIMAL(15, 2),
TotalExciseTax DECIMAL(15, 2),
FreightCost DECIMAL(15, 2),
GrossProfit DECIMAL(15, 2),
ProfitMargin DECIMAL(15, 2),
StockTurnOver DECIMAL(15, 2),
SalestoPruchaseRatio DECIMAL(15, 2),
PRIMARY KEY (VendorNumber, Brand)
);

"""

cursor.execute(query)

OperationalError: table new_vendor_sales_summary already exists

In [126]:
Query('new_vendor_sales_summary')

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchasedQuantity,TotalPurchaseDollar,TotalSalesQuantity,TotalSalesDollar,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnOver,SalestoPruchaseRatio


The final summarized DataFrame is stored into a new table `new_vendor_sales_summary` in the database.

In [131]:
final_summary.to_sql("new_vendor_sales_summary", conn, if_exists='replace', index=False)

10692

In [132]:
Query('new_vendor_sales_summary')

,VendorName,VendorNumber,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchasedQuantity,TotalPurchaseDollar,TotalSalesDollar,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnOver,SalestoPruchaseRatio
0,BROWN-FORMAN CORP,1128,1233,Jack Daniels No 7 Black,26.27,36.99,1750.0,145080,3811251.60,5101919.51,672819.31,142049.0,260999.20,68601.68,1290667.91,25.297693,0.979108,1.338647
1,MARTIGNETTI COMPANIES,4425,3405,Tito's Handmade Vodka,23.19,28.99,1750.0,164038,3804041.22,4819073.49,561512.37,160247.0,294438.66,144929.24,1015032.27,21.062810,0.976890,1.266830
2,PERNOD RICARD USA,17035,8068,Absolut 80 Proof,18.24,24.99,1750.0,187407,3418303.68,4538120.60,461140.15,187140.0,343854.07,123780.22,1119816.92,24.675786,0.998575,1.327594
3,DIAGEO NORTH AMERICA INC,3960,4261,Capt Morgan Spiced Rum,16.17,22.99,1750.0,201682,3261197.94,4475972.88,420050.01,200412.0,368242.80,257032.07,1214774.94,27.139908,0.993703,1.372493
4,DIAGEO NORTH AMERICA INC,3960,3545,Ketel One Vodka,21.89,29.99,1750.0,138109,3023206.01,4223107.62,545778.28,135838.0,249587.83,257032.07,1199901.61,28.412764,0.983556,1.396897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,WINE GROUP INC,9815,8527,Concannon Glen Ellen Wh Zin,1.32,4.99,750.0,2,2.64,15.95,10.96,5.0,0.55,27100.41,13.31,83.448276,2.500000,6.041667
10688,SAZERAC CO INC,8004,5683,Dr McGillicuddy's Apple Pie,0.39,0.49,50.0,6,2.34,65.66,1.47,134.0,7.04,50293.62,63.32,96.436186,22.333333,28.059829
10689,HEAVEN HILL DISTILLERIES,3924,9123,Deep Eddy Vodka,0.74,0.99,50.0,2,1.48,1.98,0.99,2.0,0.10,14069.87,0.50,25.252525,1.000000,1.337838
10690,DIAGEO NORTH AMERICA INC,3960,6127,The Club Strawbry Margarita,1.47,1.99,200.0,1,1.47,143.28,77.61,72.0,15.12,257032.07,141.81,98.974037,72.000000,97.469388
